# Nepali Gemma benchmark
Enable Internet and GPU T4 x2 in Kaggle. Add a `HF_TOKEN` secret after accepting each Gemma model's Hugging Face terms. E2B is the default smoke test; E4B and 12B require both T4 GPUs.

In [ ]:
!pip install -q transformers==5.10.1 accelerate==1.12.0 bitsandbytes==0.49.2 sentencepiece==0.2.1 huggingface_hub==0.36.2
import os, gc, time, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=token)
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Change only this list. Start with E2B; use T4 x2 for E4B/12B.
MODELS_TO_RUN = ['gemma4_e2b']
MODEL_IDS = {'gemma4_e2b':'google/gemma-4-E2B-it', 'gemma4_e4b':'google/gemma-4-E4B-it', 'gemma4_12b':'google/gemma-4-12B-it'}
PROMPT = 'नेपालको स्थानीय सेवाबारे ३ वाक्यमा विनम्र जानकारी दिनुहोस्। तथ्य थाहा नभए अनुमान नगर्नुहोस्।'
SYSTEM = 'तपाईं संक्षिप्त, विनम्र नेपाली सहायक हुनुहुन्छ। देवनागरी नेपालीमा उत्तर दिनुहोस्। तथ्य नबनाउनुहोस्।'
MAX_NEW_TOKENS, TEMPERATURE = 300, 0.0
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)

In [ ]:
for name in MODELS_TO_RUN:
    model_id = MODEL_IDS[name]
    if name != 'gemma4_e2b' and torch.cuda.device_count() < 2: raise RuntimeError('E4B/12B require T4 x2 for this benchmark.')
    limits = {0:'11GiB', 1:'11GiB'} if name != 'gemma4_e2b' else None
    processor = AutoProcessor.from_pretrained(model_id, token=token)
    model = AutoModelForImageTextToText.from_pretrained(model_id, token=token, quantization_config=quant, device_map='auto', max_memory=limits, torch_dtype=torch.float16)
    messages = [{'role':'system','content':SYSTEM},{'role':'user','content':PROMPT}]
    rendered = processor.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=True, tokenize=False)
    inputs = processor(text=rendered, return_tensors='pt').to(model.device)
    started=time.perf_counter(); output=model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False); elapsed=time.perf_counter()-started
    text=processor.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    print(f'\n[{name}] {elapsed:.1f}s\n{text}')
    del model, processor; gc.collect(); torch.cuda.empty_cache()